In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG =  "workspace"
SCHEMA = "nyctaxi_project"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_trips"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_trips"
REJECTED_TABLE = f"{CATALOG}.{SCHEMA}.silver_rejected_trips"

# Umbrales de calidad (documentados en README).
MAX_DURATION_MIN = 360 # 6 HORAS
MAX_DISTANCE_MI = 100
MAX_SPEED_MPH = 80

In [0]:
bronze = spark.table(BRONZE_TABLE)

typed = (
    bronze
    .select(
        F.expr("try_cast(tpep_pickup_datetime as timestamp)").alias("pickup_ts"),
        F.expr("try_cast(tpep_dropoff_datetime as timestamp)").alias("dropoff_ts"),
        F.expr("try_cast(trip_distance as double)").alias("trip_distance"),
        F.expr("try_cast(fare_amount as double)").alias("fare_amount"),
        F.expr("try_cast(pickup_zip as int)").alias("pickup_zip"),
        F.expr("try_cast(dropoff_zip as int)").alias("dropoff_zip"),
        F.col ("source_files").alias("source_file"),
        "ingestion_ts",
    )
    .withColumn("trip_id", F.sha2(F.concat_ws("|",
        F.col("pickup_ts").cast("string"), F.col("dropoff_ts").cast("string"),
        F.col("trip_distance").cast("string"), F.col("fare_amount").cast("string"),
        F.col("pickup_zip").cast("string"), F.col("dropoff_zip").cast("string")), 256))
    .withColumn("duration_min",
        (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / 60)
    .withColumn("speed_mph",
        F.when(F.col("duration_min") > 0,
               F.col("trip_distance") / (F.col("duration_min") / 60)))
    .withColumn("pickup_date", F.to_date("pickup_ts"))
    .withColumn("pickup_hour", F.hour("pickup_ts"))
    .withColumn("pickup_dow", F.date_format("pickup_ts", "E"))
)

In [0]:
w = Window.partitionBy("trip_id").orderBy("ingestion_ts")

reasons = F.array(
    F.when(F.col("pickup_ts").isNull(),  F.lit("pickup_nulo_o_invalido")),
    F.when(F.col("dropoff_ts").isNull(), F.lit("dropoff_nulo_o_invalido")),
    F.when(F.col("duration_min") <= 0,   F.lit("duracion_no_positiva")),
    F.when(F.col("duration_min") > MAX_DURATION_MIN, F.lit("duracion_excesiva")),
    F.when(F.col("trip_distance").isNull() | (F.col("trip_distance") <= 0),
           F.lit("distancia_nula_o_no_positiva")),
    F.when(F.col("trip_distance") > MAX_DISTANCE_MI, F.lit("distancia_excesiva")),
    F.when(F.col("fare_amount").isNull() | (F.col("fare_amount") <= 0),
           F.lit("tarifa_nula_o_no_positiva")),
    F.when(F.col("speed_mph") > MAX_SPEED_MPH, F.lit("velocidad_irreal")),
    F.when(F.col("pickup_zip").isNull() | F.col("dropoff_zip").isNull(),
           F.lit("zip_nulo")),
    F.when(F.row_number().over(w) > 1,   F.lit("duplicado")),
)

evaluated = (
    typed
    .withColumn("reject_reasons", F.filter(reasons, lambda x: x.isNotNull()))
    .withColumn("is_valid", F.size("reject_reasons") == 0)
)

In [0]:
(evaluated.filter("is_valid")
    .drop("reject_reasons", "is_valid")
    .write.mode("overwrite").option("overwriteSchema", True)
    .saveAsTable(SILVER_TABLE))

(evaluated.filter("NOT is_valid")
    .drop("is_valid")
    .withColumn("reject_reasons", F.concat_ws(", ", "reject_reasons"))
    .write.mode("overwrite").option("overwriteSchema", True)
    .saveAsTable(REJECTED_TABLE))

print("Silver y rechazados escritos")

Silver y rechazados escritos


In [0]:
n_bronze = spark.table(BRONZE_TABLE).count()
n_silver = spark.table(SILVER_TABLE).count()
n_rej    = spark.table(REJECTED_TABLE).count()

print(f"Bronze: {n_bronze} | Silver: {n_silver} | Rechazados: {n_rej}")
print(f"Silver + Rechazados = {n_silver + n_rej} | ¿Coincide con Bronze? {n_silver + n_rej == n_bronze}")
print(f"% rechazado: {round(100 * n_rej / n_bronze, 2)}%")

Bronze: 21932 | Silver: 21792 | Rechazados: 140
Silver + Rechazados = 21932 | ¿Coincide con Bronze? True
% rechazado: 0.64%


In [0]:
display(
    spark.table(REJECTED_TABLE)
    .select(F.explode(F.split("reject_reasons", ", ")).alias("motivo"))
    .groupBy("motivo").count().orderBy(F.desc("count"))
)

display(spark.table(SILVER_TABLE).limit(5))

motivo,count
distancia_nula_o_no_positiva,76
duracion_excesiva,32
velocidad_irreal,24
tarifa_nula_o_no_positiva,10
duracion_no_positiva,1


pickup_ts,dropoff_ts,trip_distance,fare_amount,pickup_zip,dropoff_zip,source_file,ingestion_ts,trip_id,duration_min,speed_mph,pickup_date,pickup_hour,pickup_dow
2016-02-24T18:15:01.000Z,2016-02-24T18:32:19.000Z,2.45,13.0,10199,10023,/Volumes/workspace/nyctaxi_project/raw_landing/trips/2016-02-24/part-00000-tid-5842768403752083846-fab35de7-4b73-48e2-a8c6-6834a9aafaf4-288-1-c000.csv,2026-10-12T21:25:36.536Z,0000b1d0b3b496acf17b132402ef89e3213431e696a3a4f25174c29c60b2399c,17.3,8.497109826589597,2016-02-24,18,Wed
2016-02-11T02:00:06.000Z,2016-02-11T02:06:36.000Z,1.21,6.5,10018,10153,/Volumes/workspace/nyctaxi_project/raw_landing/trips/2016-02-11/part-00000-tid-8648243490225509798-d690131a-17d1-4828-b1e5-d6ad8186655b-275-1-c000.csv,2026-10-12T21:25:36.536Z,0001db8c6a216107c69eda6941eed497ecd4b9e729cbacc51068787bca5c25ef,6.5,11.169230769230769,2016-02-11,2,Thu
2016-02-25T17:11:00.000Z,2016-02-25T17:15:20.000Z,0.63,5.0,10001,10110,/Volumes/workspace/nyctaxi_project/raw_landing/trips/2016-02-25/part-00000-tid-7412480220715647512-9d4d642e-a0f4-483f-afcf-0f1b0bf2b7e5-289-1-c000.csv,2026-10-12T21:25:36.536Z,00034d5f0c6cb98135e6350a22ad62d457a752fc9ed752e68473ac1d4914c1d1,4.333333333333333,8.723076923076924,2016-02-25,17,Thu
2016-02-19T15:40:39.000Z,2016-02-19T16:01:47.000Z,3.07,15.5,10021,10001,/Volumes/workspace/nyctaxi_project/raw_landing/trips/2016-02-19/part-00000-tid-7148610444924787464-af493d88-9855-4f9d-a389-ce6aa59da158-283-1-c000.csv,2026-10-12T21:25:36.536Z,0004b89f3eebc23381a6c319a535b92035800f018f115eeb81d4e3f4ae9c22b1,21.133333333333333,8.71608832807571,2016-02-19,15,Fri
2016-02-22T14:02:39.000Z,2016-02-22T14:11:40.000Z,1.8,8.5,10280,10012,/Volumes/workspace/nyctaxi_project/raw_landing/trips/2016-02-22/part-00000-tid-2205992018993774899-eec5bb21-c5cb-4d11-9b69-a8e02c89c2c4-286-1-c000.csv,2026-10-12T21:25:36.536Z,00058e2ef198385fdbb0c2fc0960b2d20fa811a5992b678fddc753b9cd9f724a,9.016666666666667,11.97781885397412,2016-02-22,14,Mon


In [0]:
from pyspark.sql import functions as F

P = "workspace.nyctaxi_project"
b = spark.table(f"{P}.bronze_trips")

print("Columnas:", b.columns)
col = "source_files" if "source_files" in b.columns else "source_file"

print("\nArchivos distintos:", b.select(col).distinct().count())

print("\nFilas por minuto de carga:")
(b.groupBy(F.date_trunc("minute", "ingestion_ts").alias("minuto_de_carga"))
   .count().orderBy("minuto_de_carga").show(truncate=False))

print("\nHistorial de la tabla:")
spark.sql(f"DESCRIBE HISTORY {P}.bronze_trips") \
     .select("version", "timestamp", "operation") \
     .show(30, truncate=False)

Columnas: ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance', 'fare_amount', 'pickup_zip', 'dropoff_zip', '_rescued_data', 'source_files', 'ingestion_ts']

Archivos distintos: 120

Filas por minuto de carga:
+-------------------+-----+
|minuto_de_carga    |count|
+-------------------+-----+
|2026-10-12 20:26:00|7289 |
|2026-10-12 21:25:00|14643|
|2026-10-12 21:48:00|21932|
+-------------------+-----+


Historial de la tabla:
+-------+-------------------+-----------------+
|version|timestamp          |operation        |
+-------+-------------------+-----------------+
|5      |2026-10-12 21:48:39|STREAMING UPDATE |
|4      |2026-10-12 21:25:40|STREAMING UPDATE |
|3      |2026-10-12 20:32:25|SET TBLPROPERTIES|
|2      |2026-10-12 20:26:08|STREAMING UPDATE |
|1      |2026-10-12 20:26:06|STREAMING UPDATE |
|0      |2026-10-12 20:25:56|CREATE TABLE     |
+-------+-------------------+-----------------+

